# LAB | Whisper STT Implementation
delivered by Carolina, November 2026

Scenario You're working as an AI consultant for a company that needs to transcribe meeting recordings. Your client has hours of meeting audio that need to be converted to text with timestamps for easy reference and searchability. You'll use OpenAI's Whisper API to transcribe the audio, implement chunking for long recordings, and extract timestamps for each segment. This lab will teach you both guided (with prompts) and unguided (without prompts) transcription a

In [1]:
# Step 1: Set up the environment

%pip install -q openai pydub sounddevice numpy scipy librosa matplotlib soundfile python-dotenv

import io
import os
from openai import OpenAI
import scipy.io.wavfile as wavfile
import sounddevice as sd
import numpy as np
import librosa
import librosa.display
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from IPython.display import Audio, display


#Project's file structure

for pasta in ["audio", "chunks", "outputs"]:
    os.makedirs(pasta, exist_ok=True)

print(os.listdir())

#API Key + client
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")
print("✅ Packages imported")
print("🔑 OPENAI_API_KEY: configured" if api_key else "⚠️ OPENAI_API_KEY is not configured yet")

client = OpenAI(api_key=api_key)
print("🤖 OpenAI client initialized")

Note: you may need to restart the kernel to use updated packages.
['.DS_Store', 'tts.ipynb', 'transformer_audio.ipynb', 'chunks', 'audio', '.gitignore', '.env', 'stt.ipynb', '.vscode', 'outputs', 'speech_recognition_breaking_points.ipynb', 'LAB_Whisper_Carolina.ipynb']
✅ Packages imported
🔑 OPENAI_API_KEY: configured
🤖 OpenAI client initialized


In [2]:
%pip install audioop-lts

Note: you may need to restart the kernel to use updated packages.


In [3]:
%conda install -y -c conda-forge ffmpeg

Error while loading conda entry point: conda-anaconda-telemetry (cannot import name 'CondaRequestHeader' from 'conda.plugins' (/opt/anaconda3/lib/python3.14/site-packages/conda/plugins/__init__.py))
Error while loading conda entry point: conda-anaconda-tos (cannot import name 'CondaPreCommand' from 'conda.plugins' (/opt/anaconda3/lib/python3.14/site-packages/conda/plugins/__init__.py))
Error while loading conda entry point: repo (cannot import name 'CondaSubcommand' from 'conda.plugins' (/opt/anaconda3/lib/python3.14/site-packages/conda/plugins/__init__.py))
Error while loading conda entry point: conda-anaconda-telemetry (cannot import name 'CondaRequestHeader' from 'conda.plugins' (/opt/anaconda3/lib/python3.14/site-packages/conda/plugins/__init__.py))
Error while loading conda entry point: conda-anaconda-tos (cannot import name 'CondaPreCommand' from 'conda.plugins' (/opt/anaconda3/lib/python3.14/site-packages/conda/plugins/__init__.py))
Error while loading conda entry point: repo (c

In [4]:
from pydub import AudioSegment

# Step 2: Load the sample meeting audio

audio_extensions = {".wav", ".mp3", ".m4a", ".mp4", ".mpeg", ".mpga", ".webm", ".ogg", ".flac"}
audio_files = sorted(f for f in os.listdir("audio") if not f.startswith(".") and os.path.splitext(f)[1].lower() in audio_extensions)
print("🎧 Files in audio/:", audio_files)

if not audio_files:
    raise FileNotFoundError("No supported audio files found in 'audio/'. Add an audio file and run this cell again.")

audio_path = os.path.join("audio", audio_files[0])
try:
    audio = AudioSegment.from_file(audio_path)
except Exception as exc:
    raise RuntimeError(f"Could not read '{audio_path}'. For MP3/M4A/MP4 files, make sure FFmpeg is installed.") from exc

print(f"📄 File: {audio_path}")
print(f"⏱️ Duration: {len(audio) / 1000 / 60:.1f} min")
print(f"🔊 Channels: {audio.channels}")
print(f"📈 Sample rate: {audio.frame_rate} Hz")
print(f"💾 Size: {os.path.getsize(audio_path) / 1024 / 1024:.1f} MB")

🎧 Files in audio/: ['CA138clip.mp3']
📄 File: audio/CA138clip.mp3
⏱️ Duration: 1.4 min
🔊 Channels: 1
📈 Sample rate: 44100 Hz
💾 Size: 0.9 MB


In [ ]:
# Step 3: Basic Transcription (Without Chunking)

# Cut the first 60 seconds
sample = audio[:60_000]  # pydub works in milliseconds
sample_path = os.path.join("outputs", "sample_60s.mp3")
sample.export(sample_path, format="mp3")
print(f"✂️ Sample saved: {sample_path} ({len(sample) / 1000:.0f}s)")

display(Audio(sample_path))

# Transcribe with Whisper
print("🤖 Transcribing with Whisper...")
with open(sample_path, "rb") as f:
    transcript = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="verbose_json",
    )

# Response format
print("\n📦 Response fields:", list(transcript.model_dump().keys()))
print(f"🌍 Language: {transcript.language}")
print(f"⏱️ Duration: {transcript.duration:.1f}s")
print(f"🧩 Segments: {len(transcript.segments)}")

# Basic transcription text
print("\n📝 Transcription:")
print("-" * 40)
print(transcript.text)

# First segments with timestamps
print("\n🕒 First segments:")
for seg in transcript.segments[:5]:
    print(f"[{seg.start:6.2f}s - {seg.end:6.2f}s] {seg.text}")

✂️ Sample saved: outputs/sample_60s.mp3 (60s)


🤖 Transcribing with Whisper...

📦 Response fields: ['duration', 'language', 'text', 'segments', 'usage', 'words', 'task']
🌍 Language: english
⏱️ Duration: 60.0s
🧩 Segments: 16

📝 Transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now. I think that's pretty well fact because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused, so that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up

In [7]:
# Step 4: Transcription with Prompts (Guided Approach)

# Context prompt: names, acronyms and terms
context_prompt = (
    "1969 interview recorded in Weaverville, California, the morning after "
    "the Apollo moon landing. Topics: astronauts, lunar module, command module, "
    "rendezvoused, kangaroo walk, moon rocks, soil samples, splashdown."
)

print("🤖 Transcribing WITH prompt...")
with open(sample_path, "rb") as f:
    prompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        prompt=context_prompt,
        response_format="verbose_json",
    )

print("\n💡 Prompt used:")
print(context_prompt)

print("\n📝 Prompted transcription:")
print("-" * 40)
print(prompted.text)

# Save result
prompted_path = os.path.join("outputs", "transcript_prompted.txt")
with open(prompted_path, "w", encoding="utf-8") as f:
    f.write(f"PROMPT: {context_prompt}\n\n{prompted.text}")
print(f"\n💾 Saved: {prompted_path}")


🤖 Transcribing WITH prompt...

💡 Prompt used:
1969 interview recorded in Weaverville, California, the morning after the Apollo moon landing. Topics: astronauts, lunar module, command module, rendezvoused, kangaroo walk, moon rocks, soil samples, splashdown.

📝 Prompted transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to Earth. Yes, I'll be so glad when they land back now, but I think that's pretty well a fact because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? They've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused, so that's a matter of making the circles and then comin

In [10]:
# Step 5: Transcription Without Prompts (Unguided Approach)

print("🤖 Transcribing WITHOUT prompt...")
with open(sample_path, "rb") as f:
    unguided = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="verbose_json",
    )

print("\n📝 Unguided transcription:")
print("-" * 40)
print(unguided.text)

# Save result
unguided_path = os.path.join("outputs", "transcript_unguided.txt")
with open(unguided_path, "w", encoding="utf-8") as f:
    f.write(unguided.text)

print(f"\n💾 Saved: {unguided_path}")

🤖 Transcribing WITHOUT prompt...

📝 Unguided transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now. I think that's pretty well fact because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused, so that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will.

💾 Saved: outputs/transcript_unguided.txt


In [ ]:
#COMPARISON OF PROMPTED VS UNPROMPTED TRANSCRIPTIONS

import difflib

print("🤖 Transcribing WITHOUT prompt...")
with open(sample_path, "rb") as f:
    unprompted = client.audio.transcriptions.create(
        model="whisper-1",
        file=f,
        response_format="verbose_json",
    )

print("\n📝 Unprompted transcription:")
print("-" * 40)
print(unprompted.text)

# Save result
unprompted_path = os.path.join("outputs", "transcript_unprompted.txt")
with open(unprompted_path, "w", encoding="utf-8") as f:
    f.write(unprompted.text)
print(f"\n💾 Saved: {unprompted_path}")

# --- Compare both approaches ---
print("\n🔍 COMPARISON")
print("=" * 40)

# 1. Overall similarity
similarity = difflib.SequenceMatcher(None, prompted.text, unprompted.text).ratio()
print(f"📊 Text similarity: {similarity:.1%}")

# 2. Key terms: did each version get them right?
key_terms = ["Weaverville", "rendezvous", "kangaroo", "astronaut", "moon", "rock", "soil"]
print(f"\n{'Term':15} {'Prompted':>10} {'Unprompted':>12}")
for term in key_terms:
    p = "✅" if term.lower() in prompted.text.lower() else "❌"
    u = "✅" if term.lower() in unprompted.text.lower() else "❌"
    print(f"{term:15} {p:>10} {u:>12}")

# 3. Word-level differences
print("\n✏️ Differences (- unprompted / + prompted):")
diff = difflib.ndiff(unprompted.text.split(), prompted.text.split())
changes = [d for d in diff if d.startswith(("- ", "+ "))]
print(" ".join(changes) if changes else "No differences")

🤖 Transcribing WITHOUT prompt...

📝 Unprompted transcription:
----------------------------------------
It was rather interesting just to watch them gathering their materials and bouncing around their, what do they call it, kangaroo walk or something like that. Who named it that? I don't know. I bet those men are going to get quite a reception when they get back to New York. Oh yes, I'll be so glad when they land back now. I think that's pretty well fact because they've landed so many safely now that I feel relieved. Just getting off of the moon was the thing that was... Have they met with the one that was circling? Yes, they've rendezvoused, so I understand. That wasn't shown either, but they say they have rendezvoused, so that's a matter of making the circles and then coming down. What do you sort of imagine for the future? Do you imagine them standing up? I think they will. I think they will.

💾 Saved: outputs/transcript_unprompted.txt

🔍 COMPARISON
📊 Text similarity: 97.8%

Term    

In [ ]:
#Step 6: Implementing Audio Chunking

#Split the audio into chunks of 30 seconds each
chunk_length_ms = 30_000  # 30 seconds in milliseconds
chunks = [audio[i:i + chunk_length_ms] for i in range(0, len(audio), chunk_length_ms)]

chunk_files = []

for i, start_ms in enumerate(range(0, len(audio), chunk_length_ms)):
    chunk = audio[start_ms:start_ms + chunk_length_ms]
    chunk_path = os.path.join("chunks", f"chunk_{i:03d}.mp3")
    chunk.export(chunk_path, format="mp3")
    chunk_files.append((chunk_path, start_ms / 1000))
    print(f"🔪 {chunk_path} | starts at {start_ms / 1000:.0f}s | "
          f"length {len(chunk) / 1000:.1f}s | "
          f"size {os.path.getsize(chunk_path) / 1024 / 1024:.2f} MB")

print(f"\n✅ Created {len(chunk_files)} chunks")

🔪 chunks/chunk_000.mp3 | starts at 0s | length 30.0s | size 0.23 MB
🔪 chunks/chunk_001.mp3 | starts at 30s | length 30.0s | size 0.23 MB
🔪 chunks/chunk_002.mp3 | starts at 60s | length 26.2s | size 0.20 MB

✅ Created 3 chunks


In [18]:
#Step 7: Transcribing Chunks with Timestamps

import time

def fmt_time(seconds):
    """Convert seconds to HH:MM:SS"""
    h, rem = divmod(int(seconds), 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

all_segments = []

for i, (chunk_path, offset) in enumerate(chunk_files):
    print(f"🤖 Transcribing chunk {i + 1}/{len(chunk_files)} (offset {offset:.0f}s)...")
    with open(chunk_path, "rb") as f:
        result = client.audio.transcriptions.create(
            model="whisper-1",
            file=f,
            prompt=context_prompt,          # reuse the guided prompt from Step 4
            response_format="verbose_json",
        )

    for seg in result.segments:
        all_segments.append({
            "chunk": i,
            "start": seg.start + offset,    # ⬅️ key step: add the chunk offset
            "end": seg.end + offset,
            "text": seg.text.strip(),
        })

    time.sleep(1)  # small pause to avoid rate limits

print(f"\n✅ Total segments: {len(all_segments)}")

# Combined transcript with timestamps
print("\n📝 Full transcript with timestamps:")
print("-" * 40)
for seg in all_segments:
    print(f"[{fmt_time(seg['start'])} - {fmt_time(seg['end'])}] {seg['text']}")

# Sanity check: timestamps must always go forward
ok = all(a["start"] <= b["start"] for a, b in zip(all_segments, all_segments[1:]))
print("\n✅ Timestamps are in order" if ok else "\n⚠️ Timestamps out of order — check offsets")

🤖 Transcribing chunk 1/3 (offset 0s)...
🤖 Transcribing chunk 2/3 (offset 30s)...
🤖 Transcribing chunk 3/3 (offset 60s)...

✅ Total segments: 14

📝 Full transcript with timestamps:
----------------------------------------
[00:00:00 - 00:00:13] It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that.
[00:00:13 - 00:00:15] Who named it that? I don't know.
[00:00:15 - 00:00:19] I bet those men are going to get quite a reception when they get back to Earth.
[00:00:19 - 00:00:30] Yes, I'll be so glad when they land back now, but I think that's pretty well a fact because they've landed so many safely now.
[00:00:31 - 00:00:35] Just getting off of the moon was the thing that was...
[00:00:35 - 00:00:38] Have they met with the one that was circling?
[00:00:38 - 00:00:44] Yes, they've rendezvoused. So I understand. That wasn't shown either, so I...
[00:00:44 - 00:00:53] But they say they have rendez

In [19]:
#Step 8: Exporting with Timestamps

import json

def fmt_srt_time(seconds):
    """Convert seconds to SRT format: HH:MM:SS,mmm"""
    ms = int(round(seconds * 1000))
    h, rem = divmod(ms, 3_600_000)
    m, rem = divmod(rem, 60_000)
    s, ms = divmod(rem, 1000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

base_name = os.path.splitext(os.path.basename(audio_path))[0]  # e.g. "CA138clip"

# 1. TXT — human-readable with timestamps
txt_path = os.path.join("outputs", f"{base_name}.txt")
with open(txt_path, "w", encoding="utf-8") as f:
    f.write(f"Transcript: {base_name}\n")
    f.write(f"Duration: {fmt_time(len(audio) / 1000)}\n")
    f.write("=" * 40 + "\n\n")
    for seg in all_segments:
        f.write(f"[{fmt_time(seg['start'])}] {seg['text']}\n")

# 2. SRT — subtitle format
srt_path = os.path.join("outputs", f"{base_name}.srt")
with open(srt_path, "w", encoding="utf-8") as f:
    for n, seg in enumerate(all_segments, start=1):
        f.write(f"{n}\n")
        f.write(f"{fmt_srt_time(seg['start'])} --> {fmt_srt_time(seg['end'])}\n")
        f.write(f"{seg['text']}\n\n")

# 3. JSON — structured data for search / other systems
json_path = os.path.join("outputs", f"{base_name}.json")
export = {
    "source_file": audio_path,
    "duration_seconds": round(len(audio) / 1000, 2),
    "model": "whisper-1",
    "prompt": context_prompt,
    "chunk_length_seconds": chunk_length_ms / 1000,
    "segments": all_segments,
}
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(export, f, ensure_ascii=False, indent=2)

# Checkpoint: verify files exist and are readable
for path in [txt_path, srt_path, json_path]:
    print(f"\n📄 {path} ({os.path.getsize(path)} bytes) — preview:")
    print("-" * 40)
    with open(path, encoding="utf-8") as f:
        print(f.read(300))


📄 outputs/CA138clip.txt (1309 bytes) — preview:
----------------------------------------
Transcript: CA138clip
Duration: 00:01:26

[00:00:00] It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that.
[00:00:13] Who named it that? I don't know.
[00:00:

📄 outputs/CA138clip.srt (1539 bytes) — preview:
----------------------------------------
1
00:00:00,000 --> 00:00:13,000
It was rather interesting just to watch them gathering their materials and bouncing around their, what they call it, kangaroo walk or something like that.

2
00:00:13,000 --> 00:00:15,000
Who named it that? I don't know.

3
00:00:15,000 --> 00:00:19,000
I bet those me

📄 outputs/CA138clip.json (2674 bytes) — preview:
----------------------------------------
{
  "source_file": "audio/CA138clip.mp3",
  "duration_seconds": 86.2,
  "model": "whisper-1",
  "prompt": "1969 interview recorded in Weaverville, California, the morni